<a href="https://colab.research.google.com/github/chinh-hoangduc/QuantEcon-exercises/blob/main/sandbox_misallocation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [54]:
!pip install jax

In [55]:
from typing import NamedTuple
import numpy as np
from scipy.stats import multivariate_normal, norm
import matplotlib.pyplot as plt
import jax.numpy as jnp
import jax.scipy as jsp
import jax
import quantecon as qe
from functools import partial

In [ ]:
class Household(NamedTuple):
    T: int
    a_grid: jnp.ndarray
    θ_grid: jnp.ndarray
    P_grid: jnp.ndarray
    endow_wts: jnp.ndarray



def create_household(
    T: int = 22,
    a_min: float = 0, a_num: int = 150,
    θ_num: int = 9, P_num: int = 25,
    σ_θ: float = 0.4, σ_P: float = 0.5, ρ: float = 0.4,
    n_std: int = 4,
) -> Household:

    a_max = jnp.exp(n_std * σ_P) + 1   # align wealth grid of generations
    a_grid = jnp.linspace(a_min, a_max, a_num)


    def create_endowment_grid(θ_num, P_num, σ_θ, σ_P, ρ, n_std):
        # 1. Marginal nodes: equally spaced in log space (over ±n_std * sigma)
        log_θ = jnp.linspace(-n_std * σ_θ, n_std * σ_θ, θ_num)
        log_P = jnp.linspace(-n_std * σ_P, n_std * σ_P, P_num)

        # 2. Cell boundaries: midpoints, extended to ±inf at the tails
        def edges(x):
            mids = jnp.asarray(0.5 * (x[:-1] + x[1:]))
            return jnp.concatenate([jnp.array([-jnp.inf]), mids, jnp.array([jnp.inf])])
        e_θ = edges(log_θ)   # length n_theta + 1
        e_P = edges(log_P)   # length n_P + 1

        # 3. Cell probabilities from the bivariate normal CDF
        Sigma = np.array([[σ_θ**2, ρ * σ_θ * σ_P],
                          [ρ * σ_θ * σ_P, σ_P**2]])
        rv = multivariate_normal(mean=[0.0, 0.0], cov=Sigma)

        # CDF at each corner, then inclusion–exclusion for the rectangle mass
        T, P = np.meshgrid(e_θ, e_P, indexing='ij')          # (n_theta+1, n_P+1)
        C = rv.cdf(np.stack([T.ravel(), P.ravel()], axis=-1)).reshape(T.shape)
        W = C[1:,1:] - C[:-1,1:] - C[1:,:-1] + C[:-1,:-1]        # (n_theta, n_P)
        W = W / W.sum()

        # 4. Return levels (not logs) plus the joint weight matrix
        θ_grid = jnp.asarray(np.exp(log_θ))
        P_grid = jnp.asarray(np.exp(log_P))
        endow_wts = jnp.asarray(W)
        return θ_grid, P_grid, endow_wts

    θ_grid, P_grid, endow_wts = create_endowment_grid(θ_num, P_num, σ_θ, σ_P, ρ, n_std)


    return Household(T, a_grid, θ_grid, P_grid, endow_wts )

In [ ]:
γ = 1.5

def bequest(a, abar = 5.0, ψ = 2.0):
    return ψ * (a + abar)**(1 - γ) / (1 - γ)

def u(c):
    return c**(1 - γ) /

In [89]:
# @title
# Sanity check

θ_grid, P_grid, endow_wts = create_endowment_grid()

log_theta = np.log(θ_grid)   # shape (9,)
log_P     = np.log(P_grid)   # shape (25,)
W = np.asarray(endow_wts)    # shape (9, 25)

# Marginals of the joint weight matrix
w_theta = W.sum(axis=1)      # shape (9,)
w_P     = W.sum(axis=0)      # shape (25,)

# Weighted means
mu_theta = (w_theta * log_theta).sum()
mu_P     = (w_P * log_P).sum()

# Weighted variances (marginal)
var_theta = (w_theta * (log_theta - mu_theta)**2).sum()
var_P     = (w_P * (log_P - mu_P)**2).sum()

# Weighted covariance (needs the joint)
LT, LP = np.meshgrid(log_theta, log_P, indexing='ij')
cov = (W * (LT - mu_theta) * (LP - mu_P)).sum()
rho_hat = cov / np.sqrt(var_theta * var_P)

print(f"mu:    {mu_theta:.4f}, {mu_P:.4f}     (target 0, 0)")
print(f"sigma: {np.sqrt(var_theta):.4f}, {np.sqrt(var_P):.4f}  (target 0.40, 0.50)")
print(f"rho:   {rho_hat:.4f}                 (target 0.40)")

mu:    0.0000, -0.0000     (target 0, 0)
sigma: 0.4163, 0.5023  (target 0.40, 0.50)
rho:   0.3825                 (target 0.40)


In [ ]:
class Shock(NamedTuple):
    ε_grid: jnp.ndarray
    Π: jnp.ndarray
    μ_ε: jnp.ndarray

def create_shock(ρ: float, σ_η: float, n: int = 7) -> Shock:
    mc = qe.markov.approximation.tauchen(n, ρ, σ_η)
    Π = jnp.asarray(mc.P)
    ε_grid = jnp.asarray(mc.state_values)

    # Stationary distribution: left eigenvector of Pi with eigenvalue 1
    μ_ε = jnp.asarray(mc.stationary_distributions[0])
    return Shock(ε_grid, Π, μ_ε)

In [ ]:
shock_hs = create_shock(ρ=0.8836, σ_η=0.2744, n=7)
shock_col = create_shock(ρ=0.9216, σ_η=0.2356, n=7)